# Eigenshirts: PCA on jerseys vs shirts, then OLS vs log-loss classifiers

Click **Runtime → Run all**. Every figure is shown inline and also saved as a PNG in the
Files panel on the left (folder icon), ready to download for the upload question.

If the automatic download fails, the first cell tells you to download the zips in your browser
and drag them into the Files panel; then run all again.

In [ ]:
# 1. Download and unzip the data
import io, shutil, tarfile, zipfile
from pathlib import Path
import requests

URLS = {
    "jerseys": "https://course-resources.minerva.kgi.edu/uploaded_files/mke/YpANOV/jerseys.zip",
    "shirts":  "https://course-resources.minerva.kgi.edu/uploaded_files/mke/rQb8dn/shirts.zip",
}
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36"}

def extract_all(archive, dest):
    """Extract a zip or tar into dest, then keep extracting any archives found inside it."""
    dest.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as t:
            t.extractall(dest)
    for inner in list(dest.rglob("*")):
        if inner.is_file() and inner != archive and (zipfile.is_zipfile(inner) or tarfile.is_tarfile(inner)):
            extract_all(inner, inner.parent / (inner.stem + "_extracted"))
            inner.unlink()

for name, url in URLS.items():
    zpath = Path(f"{name}.zip")
    if not (zpath.exists() and zipfile.is_zipfile(zpath)):
        print(f"downloading {name} ...")
        try:
            r = requests.get(url, headers=HEADERS, timeout=300)
            print(f"  HTTP {r.status_code}, {len(r.content) / 1e6:.1f} MB, type {r.headers.get('content-type')}")
            zpath.write_bytes(r.content)
        except Exception as e:
            print("  download error:", e)
    if not (zpath.exists() and zipfile.is_zipfile(zpath)):
        start = zpath.read_bytes()[:150] if zpath.exists() else b""
        raise RuntimeError(
            f"{zpath} is not a valid zip (starts with {start!r}).\n"
            f"Open {url} in your browser, then drag the downloaded {zpath} into the Colab "
            f"Files panel (folder icon, left) and run all again.")
    shutil.rmtree(name, ignore_errors=True)
    extract_all(zpath, Path(name))
    files = [p for p in Path(name).rglob("*") if p.is_file()]
    print(f"{name}: {len(files)} files, e.g. {[str(p) for p in files[:3]]}")


In [ ]:
# 2. Load, crop and resize every image, then PCA
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.decomposition import PCA

SIZE = 64                                  # every image becomes 64 x 64 x 3 = 12,288 numbers
FOLDERS = {"jerseys": 0, "shirts": 1}      # folder name -> label
MAX_PER_CLASS = 150                        # 150 + 150 = 300 images (>= 200 required)

def load_image(path):
    img = Image.open(path).convert("RGB")
    img = ImageOps.fit(img, (SIZE, SIZE))  # center crop to a square, then resize
    return np.asarray(img, dtype=np.float32) / 255.0

images, labels = [], []
for folder, label in FOLDERS.items():
    # Try every file: ImageNet files may have odd or missing extensions
    paths = sorted(p for p in Path(folder).rglob("*")
                   if p.is_file() and "__MACOSX" not in p.parts and not p.name.startswith("."))
    n_ok = 0
    for p in paths:
        if n_ok == MAX_PER_CLASS:
            break
        try:
            images.append(load_image(p))
            labels.append(label)
            n_ok += 1
        except Exception:
            pass                           # not an image or broken file
    print(f"{folder}: loaded {n_ok} images out of {len(paths)} files")
    if n_ok == 0:
        raise RuntimeError(f"No readable images in '{folder}'. Files found: {[str(p) for p in paths[:10]]}")

images = np.array(images)
y = np.array(labels)
X = images.reshape(len(images), -1)        # flatten: one row per image
print("images:", images.shape, " X:", X.shape, " per class:", np.bincount(y))

# How many eigenvectors? Smallest k that keeps 90% of the variance
pca_full = PCA().fit(X)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
k = int(np.argmax(cum_var >= 0.90) + 1)
for n in [10, 50, 100]:
    if n <= len(cum_var):
        print(f"{n:>4} components keep {cum_var[n - 1]:.1%} of the variance")
print(f"components for 90% variance: k = {k}  (out of {X.shape[1]} original dimensions)")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(cum_var) + 1), cum_var)
plt.axhline(0.90, color="r", linestyle="--", label="90% variance")
plt.axvline(k, color="r", linestyle=":", label=f"k = {k}")
plt.xlabel("number of components")
plt.ylabel("cumulative explained variance")
plt.title("How many eigenvectors to keep")
plt.legend()
plt.savefig("1_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 3. Visualizations: originals, eigenshirts, reconstructions, PC scatter
def to_img(v):
    v = v.reshape(SIZE, SIZE, 3)
    return (v - v.min()) / (v.max() - v.min())   # rescale to 0-1 so it displays

# Sample of the cropped and resized originals
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
rng = np.random.RandomState(1)
for row, (label, name) in enumerate([(0, "jersey"), (1, "shirt")]):
    for col, i in enumerate(rng.choice(np.where(y == label)[0], 6, replace=False)):
        axes[row, col].imshow(images[i])
        axes[row, col].set_title(name)
        axes[row, col].axis("off")
fig.suptitle(f"Cropped and resized originals ({SIZE}x{SIZE})")
plt.savefig("2_originals.png", dpi=150, bbox_inches="tight")
plt.show()

# Mean image and the first eigenshirts
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes[0, 0].imshow(pca_full.mean_.reshape(SIZE, SIZE, 3))
axes[0, 0].set_title("mean")
for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(to_img(pca_full.components_[i]))
    ax.set_title(f"PC {i + 1} ({pca_full.explained_variance_ratio_[i]:.1%})")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("Mean image and eigenshirts")
plt.savefig("3_eigenshirts.png", dpi=150, bbox_inches="tight")
plt.show()

# What the images look like in the reduced representation
def reconstruct(n):
    p = PCA(n).fit(X)
    return np.clip(p.inverse_transform(p.transform(X)), 0, 1)

levels = [5, 20, k]
recs = {n: reconstruct(n) for n in levels}
idx = np.random.RandomState(0).choice(len(X), 6, replace=False)
fig, axes = plt.subplots(len(levels) + 1, 6, figsize=(12, 2.2 * (len(levels) + 1)))
for col, i in enumerate(idx):
    axes[0, col].imshow(images[i])
    for row, n in enumerate(levels, start=1):
        axes[row, col].imshow(recs[n][i].reshape(SIZE, SIZE, 3))
axes[0, 0].set_ylabel("original")
for row, n in enumerate(levels, start=1):
    axes[row, 0].set_ylabel(f"{n} PCs")
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Images rebuilt from the reduced representation")
plt.savefig("4_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

# The data in the first two principal components
Z = pca_full.transform(X)
plt.figure(figsize=(6, 5))
for label, name in [(0, "jerseys"), (1, "shirts")]:
    plt.scatter(Z[y == label, 0], Z[y == label, 1], alpha=0.6, label=name)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Images projected onto the first two PCs")
plt.legend()
plt.savefig("5_pc_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#OLS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Same split for both classifiers, stratified so both classes are balanced
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
COMPONENTS = [n for n in [2, 5, 10, 20, 50] if n < k] + [k, None]   # None = raw pixels

def project(n_components):
    """PCA fitted on the training images only, so no test information leaks in."""
    if n_components is None:
        return X_train, X_test
    pca = PCA(n_components).fit(X_train)
    return pca.transform(X_train), pca.transform(X_test)

def ols_classifier(n_components):
    Z_train, Z_test = project(n_components)
    ols = LinearRegression().fit(Z_train, y_train)   # minimizes sum of squared errors
    pred_train = (ols.predict(Z_train) >= 0.5).astype(int)
    pred_test = (ols.predict(Z_test) >= 0.5).astype(int)
    return accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test), pred_test

print("OLS (squared loss) classifier")
print(f"{'components':>12} {'train acc':>10} {'test acc':>10}")
ols_test = []
for n in COMPONENTS:
    tr, te, _ = ols_classifier(n)
    ols_test.append(te)
    print(f"{'raw pixels' if n is None else n:>12} {tr:>10.3f} {te:>10.3f}")

_, _, pred_k = ols_classifier(k)
print(f"\nConfusion matrix with k = {k} components (rows = true jersey/shirt, cols = predicted):")
print(confusion_matrix(y_test, pred_k))


In [ ]:
#Log Loss
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

def logistic_classifier(n_components):
    Z_train, Z_test = project(n_components)
    clf = LogisticRegression(max_iter=5000).fit(Z_train, y_train)   # minimizes log loss
    p_test = clf.predict_proba(Z_test)[:, 1]
    return (accuracy_score(y_train, clf.predict(Z_train)),
            accuracy_score(y_test, clf.predict(Z_test)),
            log_loss(y_test, p_test),
            clf.predict(Z_test))

print("Logistic regression (log loss) classifier")
print(f"{'components':>12} {'train acc':>10} {'test acc':>10} {'test logloss':>13}")
log_test = []
for n in COMPONENTS:
    tr, te, ll, _ = logistic_classifier(n)
    log_test.append(te)
    print(f"{'raw pixels' if n is None else n:>12} {tr:>10.3f} {te:>10.3f} {ll:>13.3f}")

_, _, _, pred_k = logistic_classifier(k)
print(f"\nConfusion matrix with k = {k} components (rows = true jersey/shirt, cols = predicted):")
print(confusion_matrix(y_test, pred_k))

# Compare both losses
names = [("raw" if n is None else str(n)) for n in COMPONENTS]
xs = np.arange(len(names))
plt.figure(figsize=(7, 4))
plt.plot(xs, ols_test, "o-", label="OLS (squared loss)")
plt.plot(xs, log_test, "s-", label="Logistic (log loss)")
plt.xticks(xs, names)
plt.xlabel("number of PCA components")
plt.ylabel("test accuracy")
plt.title("Jerseys vs shirts: test accuracy by representation")
plt.legend()
plt.savefig("6_classifier_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
